## Topic 5: RecursiveCharacterTextSplitter Mechanics

### Concept, Intuition, Why It Exists

- CharacterTextSplitter (Topic 4) has one weakness baked into its design: it commits to a *single* separator, and the instant that separator doesn't appear within the size budget, it falls back straight to a hard character cut — there's no middle ground between "clean separator-based split" and "blind character chop."
- **RecursiveCharacterTextSplitter** fixes exactly this by trying a *list* of separators, in priority order, instead of just one — falling back progressively from the most meaning-preserving separator to the least, rather than jumping straight from "best case" to "worst case."
- The default separator priority order is: **paragraph break → line break → space → character**. This order isn't arbitrary — it's a deliberate ranking from "most likely to preserve a complete idea" to "least likely," and the splitter only drops down a level when the current level genuinely can't produce a small-enough piece.
- This is the same cost-ordering philosophy seen repeatedly across earlier topics — hash-before-embedding in dedup, structure-aware-before-generic in chunking strategy choice — applied here as: try-the-best-option-before-falling-back, at every single split decision, not just once globally.

### Internal Working

The splitter processes text **recursively**, one separator level at a time:

1. Start with the first separator in the priority list — paragraph break (typically a double newline).
2. Split the text on that separator. For each resulting piece: if it already fits within `chunk_size`, keep it as a chunk and move on. If it's still too large, **don't fall back to a hard cut immediately** — instead, recursively re-apply the splitting process to that oversized piece using the *next* separator down the priority list.
3. This continues down the chain: a paragraph too big gets split by line breaks; a line too big gets split by spaces (i.e., word boundaries); a single "word" or run of text too big even for that (rare, but possible with no whitespace at all) finally gets hard-cut by character count as the absolute last resort.
4. Adjacent small pieces produced by a given separator level get **merged back together** up to `chunk_size` before moving on — the splitter doesn't emit one chunk per individual line or sentence if several of them comfortably fit together under the size limit; it greedily packs them, the same accumulate-until-the-limit logic Topic 2's sentence-aware chunker uses, just generalized across separator levels.

### Why Paragraph → Line → Space → Character Is the Default Order

- **Paragraph break first**: a paragraph is, in most well-written prose, the largest unit that's still reliably a complete, coherent idea — splitting here first means the *majority* of real documents never need to fall further down the chain at all.
- **Line break second**: when a paragraph alone is still too large (long policy text, dense technical content), individual lines are the next-most-coherent fallback — closer to a complete thought than an arbitrary word-count cut, especially for structured content like bullet points or numbered clauses that are often one per line.
- **Space (word boundary) third**: if even a single line is too long, splitting on spaces guarantees the cut never lands mid-word — it sacrifices sentence/line coherence but still preserves the smallest meaningful linguistic unit, a word.
- **Character last, absolute fallback**: only triggered when a single "word" (or a run of text with no whitespace at all — a long ID string, a URL, a block of code with no spaces) is itself larger than the chunk size. This is the only level where a cut can land somewhere genuinely meaningless, and it's reached only when every more meaning-preserving option has already failed.
- The order is, in short, a ranked bet: try the split that's most likely to preserve meaning first, and only pay the cost of a worse split when the better option provably doesn't work for that specific piece of text.

### How It's Implemented In This Project

- This project's hand-written sentence-aware chunker (Topic 2) is a simpler, single-level version of this same idea — it commits to "sentence" as its one splitting unit, the way CharacterTextSplitter commits to one separator, without the recursive fallback chain.
- Adopting full recursive splitting becomes worthwhile the moment source content stops being reliably well-formed prose — e.g. ingesting raw SOP text with inconsistent paragraph spacing, where sentence-only splitting might occasionally hit a single "sentence" (really an unbroken run of clauses with no terminal punctuation) too large to fit the size budget, with nowhere to fall back to.

### Real-World Issues, Edge Cases, Debugging

- **Recursion depth matters for content with unusual structure**: code blocks, tables rendered as plain text, or extremely long unbroken strings (IDs, URLs) can force the splitter all the way down to the character-level fallback — worth explicitly checking which separator level actually produced each chunk in production, since chunks from the character-level fallback are exactly the syntactically-broken pieces every other strategy in this sub-chapter has been trying to avoid.
- **Merging behavior can still produce topically-mixed chunks**: because adjacent small pieces get packed together up to the size limit, two genuinely unrelated short paragraphs can still end up merged into one chunk purely because they fit — this splitter solves the "broken mid-sentence" problem completely, but doesn't solve the "topically blurry chunk" problem any better than plain sentence-aware chunking does; that's still semantic chunking's job if needed.
- **Custom separator lists for non-English or structured content**: the default paragraph/line/space/character order assumes whitespace-delimited language with clear paragraph breaks — content in a language without word-spacing, or content that's mostly tabular/structured rather than prose, needs a custom separator priority list rather than relying on defaults built around English-prose assumptions.

### Design Decisions & Trade-offs

- Recursive fallback vs. single-separator-then-hard-cut: more implementation complexity (a recursive function instead of a single split-and-pack loop) in exchange for meaningfully fewer character-level hard cuts across a real, varied corpus — worth the complexity the moment source content isn't uniformly well-formed.
- Default separator order vs. custom order per content type: the default order is a reasonable general-purpose bet for English prose, but isn't universally correct — code, tables, and non-English content may need their own tuned separator priority lists, the same per-source-type tuning principle seen throughout this sub-chapter.

### Alternatives & When To Use Each

- CharacterTextSplitter (Topic 4) — content is reliably well-formed with a consistent separator throughout, and the rare hard-cut fallback case is acceptable.
- RecursiveCharacterTextSplitter (this topic) — general-purpose default for mixed or imperfectly-formed prose; meaningfully reduces hard-cut fallback frequency at the cost of slightly more complex logic.
- Structure-aware splitting (Topic 3/4) — still the better choice whenever the content has known, reliable structure to exploit directly; recursive character splitting is a strong *fallback* for everything else, not a replacement for structure-aware splitting where structure genuinely exists.

### Common Mistakes & Production Failures

- Assuming RecursiveCharacterTextSplitter solves topical coherence because it sounds more sophisticated than sentence-aware chunking — it solves syntactic breakage far better, but says nothing about whether merged-together pieces are actually about the same topic.
- Using the default separator order unmodified on non-prose content (code, tabular text, non-English text without paragraph conventions) and getting worse results than a simpler, content-tuned splitter would have produced.
- Never checking which separator level actually produced a given chunk in production, missing a real signal that some documents are forcing frequent character-level fallback and need a different handling path entirely.

### Lead-Level Interview Questions

**Q: Why try multiple separators in priority order instead of just picking the one separator most likely to work, like CharacterTextSplitter does?**
A: No single separator is reliably present across every document and every piece of a document — a paragraph break might be missing in one section but present in another. Trying separators in priority order, recursively, means the splitter only pays the cost of a worse cut on the *specific pieces* that actually need it, rather than committing the entire document to one separator's worst case.

**Q: Walk through what happens when a single paragraph is too large even after splitting on line breaks and spaces.**
A: The splitter falls all the way to the character-level fallback for that specific piece, hard-cutting it regardless of word boundaries. This only happens for genuinely pathological content — an extremely long unbroken string with no internal whitespace at all (a URL, an ID, a run-on technical string) — and it's the one level where a cut can land somewhere truly meaningless, which is exactly why it's the last resort rather than an early option.

**Q: Does using RecursiveCharacterTextSplitter mean you no longer need to worry about topically-mixed chunks?**
A: No — it solves a different problem. It guarantees chunks aren't broken mid-word/mid-line/mid-paragraph more often than necessary, but two short, unrelated paragraphs can still get packed into the same chunk simply because they fit the size budget together. Topical coherence is what semantic chunking specifically targets; recursive character splitting and semantic chunking solve different problems and can be combined, not substituted for each other.

**Q: A new source type uses tables rendered as plain text with no real paragraph breaks. Would you use the default separator order?**
A: No — the default order assumes prose with reliable paragraph/line/space structure, which tabular plain text doesn't have. A custom separator priority list tuned to that content's actual structure (e.g. splitting on row delimiters first) would produce far better results than forcing the English-prose-tuned default order onto fundamentally non-prose content.

### Hidden Concepts Worth Knowing

- **Recursion as a general fallback-chain pattern**: trying the best option first and recursively degrading to progressively worse-but-more-reliable options is a pattern that shows up well beyond chunking — retry logic with escalating strategies, graceful degradation in distributed systems, and fallback model routing (try the best model, fall back to a cheaper/faster one on failure) all follow the same shape.
- **Separator order is itself a tunable hyperparameter, not a fixed law**: nothing about "paragraph → line → space → character" is mathematically required — it's an empirically reasonable default for English prose, and treating it as the one correct order rather than a sensible starting point is a subtle trap.

### Revision Summary

> RecursiveCharacterTextSplitter improves on single-separator splitting by trying a priority-ordered list of separators — paragraph, then line, then space, then character — recursively falling back only when the current level genuinely can't produce a small-enough piece. The order is a deliberate bet: try the most meaning-preserving split first, pay for a worse one only when necessary. It solves syntactic breakage well, but not topical coherence — that remains semantic chunking's job, and the two are complementary, not substitutes.

In [1]:
"""
recursive_text_splitter.py
-----------------------------
RecursiveCharacterTextSplitter mechanics: tries separators in priority
order -- paragraph -> line -> space -> character -- recursively falling
back to the next separator only when the current one can't produce a
small-enough piece.

Unlike CharacterTextSplitter (Topic 4), which commits to ONE separator and
hard-cuts the instant it fails, this tries progressively less-preserving
splits before ever resorting to a blind character cut.
"""

from document import make_document

DEFAULT_SEPARATORS = ["\n\n", "\n", " ", ""]  # paragraph, line, space, character


def _split_with_separator(text: str, separator: str) -> list:
    if separator == "":
        return list(text)  # character-level: every character is its own piece
    return text.split(separator)


def _recursive_split(text: str, separators: list, chunk_size: int) -> list:
    """Core recursive logic: try the first separator; for any piece still
    too large, recurse into it with the NEXT separator down the list."""
    if len(text) <= chunk_size:
        return [text]

    separator = separators[0]
    remaining_separators = separators[1:]
    pieces = _split_with_separator(text, separator)

    final_pieces = []
    for piece in pieces:
        if len(piece) <= chunk_size:
            final_pieces.append(piece)
        elif remaining_separators:
            # this piece is still too big -- recurse with the next separator
            final_pieces.extend(_recursive_split(piece, remaining_separators, chunk_size))
        else:
            # no separators left -- absolute last resort, hard character cut
            final_pieces.extend(
                piece[i:i + chunk_size] for i in range(0, len(piece), chunk_size)
            )

    return final_pieces


def _merge_small_pieces(pieces: list, separator: str, chunk_size: int) -> list:
    """Greedily packs adjacent small pieces back together up to chunk_size,
    so we don't emit one chunk per tiny line/sentence when several fit
    comfortably together."""
    merged, current = [], ""
    for piece in pieces:
        candidate = (current + separator + piece) if current else piece
        if len(candidate) <= chunk_size:
            current = candidate
        else:
            if current:
                merged.append(current)
            current = piece
    if current:
        merged.append(current)
    return merged


def recursive_character_split(text: str, chunk_size: int = 200, separators: list = None) -> list:
    separators = separators or DEFAULT_SEPARATORS
    raw_pieces = _recursive_split(text, separators, chunk_size)
    # merge using the broadest separator that's still safe (paragraph break)
    return _merge_small_pieces(raw_pieces, separators[0], chunk_size)


if __name__ == "__main__":
    sample = (
        "Premature withdrawal incurs a 1 percent penalty on the applicable rate.\n\n"
        "This does not apply if the FD is closed due to the death of the depositor. "
        "In such cases, the full contracted interest rate is paid up to the date of closure.\n\n"
        "Senior citizens receive an additional 0.5 percent interest on all tenures. "
        "This additional rate applies only to resident senior citizens aged 60 and above."
    )

    print("--- Default separator order (paragraph -> line -> space -> char) ---")
    for i, c in enumerate(recursive_character_split(sample, chunk_size=120)):
        print(f"  Chunk {i} ({len(c)} chars): {c!r}")

    print("\n--- Custom separator order (forces deeper fallback for comparison) ---")
    for i, c in enumerate(recursive_character_split(sample, chunk_size=40, separators=["\n\n", " ", ""])):
        print(f"  Chunk {i} ({len(c)} chars): {c!r}")

--- Default separator order (paragraph -> line -> space -> char) ---
  Chunk 0 (120 chars): 'Premature withdrawal incurs a 1 percent penalty on the applicable rate.\n\nThis\n\ndoes\n\nnot\n\napply\n\nif\n\nthe\n\nFD\n\nis\n\nclosed'
  Chunk 1 (120 chars): 'due\n\nto\n\nthe\n\ndeath\n\nof\n\nthe\n\ndepositor.\n\nIn\n\nsuch\n\ncases,\n\nthe\n\nfull\n\ncontracted\n\ninterest\n\nrate\n\nis\n\npaid\n\nup\n\nto\n\nthe'
  Chunk 2 (110 chars): 'date\n\nof\n\nclosure.\n\nSenior\n\ncitizens\n\nreceive\n\nan\n\nadditional\n\n0.5\n\npercent\n\ninterest\n\non\n\nall\n\ntenures.\n\nThis'
  Chunk 3 (86 chars): 'additional\n\nrate\n\napplies\n\nonly\n\nto\n\nresident\n\nsenior\n\ncitizens\n\naged\n\n60\n\nand\n\nabove.'

--- Custom separator order (forces deeper fallback for comparison) ---
  Chunk 0 (35 chars): 'Premature\n\nwithdrawal\n\nincurs\n\na\n\n1'
  Chunk 1 (37 chars): 'percent\n\npenalty\n\non\n\nthe\n\napplicable'
  Chunk 2 (38 chars): 'rate.\n\nThis\n\ndoes\n\nnot\n\napply\n\nif\n\nthe'
 